# Toronto Airbnb Listings: Modelling Price Determinants and Testing for Structural Change Under Municipal Short-Term Rental Licensing

**MMA 860 — The Acquisition and Management of Data**
**Author:** Team Isabel

---


> **Note:** Charts in this notebook are interactive Plotly figures. They were cleared to keep the file small enough for GitHub to preview, so run the notebook to see them. Static versions are in the [`figures/`](../figures) folder, and the full write-up is in the [report](../report/Toronto_Airbnb_Pricing_Report.pdf). Download the data first (see [`data/README.md`](../data/README.md)).

## 1. Setup and Imports

In [1]:
import pandas as pd
import numpy as np
import os.path as osp
import json
import warnings

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy import stats

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

# Consistent color palette used throughout the notebook
COLOR_MAIN = '#1F4E79'
COLOR_ACCENT = '#C0504D'
PLOTLY_TEMPLATE = 'plotly_white'

np.random.seed(42)  # for the train/test split

## 2. Data Import

Three Inside Airbnb files plus the neighbourhood boundary file. `listings.csv` is the detailed, listing-level export (90 fields); `calendar.csv` is a date-level availability file; `reviews.csv` is review-level text data. `calendar.csv` in this snapshot contains **availability only, not per-night price**.

In [2]:
DATA_DIR = osp.join(osp.pardir, 'data')  # raw files go in the repo's data/ folder

listings = pd.read_csv(osp.join(DATA_DIR, 'listings.csv'))
calendar = pd.read_csv(osp.join(DATA_DIR, 'calendar.csv.gz'))
reviews = pd.read_csv(osp.join(DATA_DIR, 'reviews.csv'))

with open(osp.join(DATA_DIR, 'neighbourhoods.geojson')) as f:
    neighbourhoods_geo = json.load(f)

print('listings:', listings.shape)
print('calendar:', calendar.shape)
print('reviews: ', reviews.shape)
print('neighbourhood polygons:', len(neighbourhoods_geo['features']))

listings: (15869, 90)
calendar: (8112502, 5)
reviews:  (576085, 6)
neighbourhood polygons: 140


In [3]:
listings.dtypes.value_counts()

float64    37
str        29
int64      24
Name: count, dtype: int64

## 3. Data Cleaning and Preparation

### 3.1 Cleaning the `price` field

`price` arrives as text (e.g. `"$99.85"`). We strip the `$` and any thousands separators and convert to numeric.

In [4]:
listings['price'] = (
    listings['price'].astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
)
listings['price'] = pd.to_numeric(listings['price'], errors='coerce')

print('Price summary:')
print(listings['price'].describe().round(2))
print('\nMissing price:', listings['price'].isna().sum(), f"({listings['price'].isna().mean()*100:.1f}%)")

Price summary:
count   15,008.000
mean       275.080
std        359.140
min          6.080
25%        100.620
50%        193.000
75%        357.000
max     11,042.360
Name: price, dtype: float64

Missing price: 861 (5.4%)


**Real finding:** mean price CA275 is well above the median CA193, and the maximum CA$11,042/night is over 50x the median. This is a strongly right-skewed distribution.

### 3.2 Missing data audit

In [5]:
candidate_cols = ['price', 'accommodates', 'bedrooms', 'beds', 'bathrooms', 'bathrooms_text',
                  'room_type', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed',
                  'host_is_superhost', 'hosts_time_as_host_years', 'review_scores_rating',
                  'minimum_nights', 'instant_bookable', 'license']

missing_pct = listings[candidate_cols].isna().mean().sort_values(ascending=False) * 100
missing_pct.to_frame('% missing').round(1)

,% missing
neighbourhood_group_cleansed,100.000
instant_bookable,100.000
bathrooms,54.000
beds,52.400
license,35.300
bedrooms,25.700
review_scores_rating,17.900
price,5.400
bathrooms_text,0.200
minimum_nights,0.000


**Real findings from this audit:**

- `neighbourhood_group_cleansed` is **100% missing** for Toronto. We shall drop it.
- `instant_bookable` is **also 100% missing**. We drop it too.
- `bathrooms` (the numeric field) is missing more than half the time, but `bathrooms_text` (e.g. *"1.5 baths"*) is populated for nearly every listing — we'll parse the text field instead of imputing the numeric one from scratch.
- `bedrooms` (26%) and `beds` (52%) have real, substantial missingness and need imputation.
- `review_scores_rating` (18%) is missing mainly for listings with no reviews yet. This is a MAR pattern, not something to impute blindly with the sample mean without flagging it.

### 3.3 Dropping fields that cannot be used

Both fields identified above as fully missing are dropped here, with an assertion that documents *why* — so this doesn't look like an accidental deletion if someone re-runs this on a future snapshot where the fields might actually be populated.

In [6]:
assert listings['neighbourhood_group_cleansed'].isna().all(), \
    'neighbourhood_group_cleansed is not fully missing anymore -- reconsider dropping it'
assert listings['instant_bookable'].isna().all(), \
    'instant_bookable is not fully missing anymore -- reconsider dropping it'

listings = listings.drop(columns=['neighbourhood_group_cleansed', 'instant_bookable'])
print('Columns remaining:', listings.shape[1])

Columns remaining: 88


### 3.4 Parsing `bathrooms_text`

`bathrooms_text` holds values like `"1 bath"`, `"1.5 baths"`, `"1 private bath"`, `"Half-bath"`. We extract the numeric value, treating an unqualified "half-bath" as 0.5.

In [7]:
import re

def parse_bathrooms(text):
    if pd.isna(text):
        return np.nan
    text = str(text).lower()
    if 'half' in text:
        return 0.5
    match = re.search(r'[\d.]+', text)
    return float(match.group()) if match else np.nan

listings['bathrooms_clean'] = listings['bathrooms_text'].apply(parse_bathrooms)

print('Missing after parsing bathrooms_text:', listings['bathrooms_clean'].isna().sum(),
      f"({listings['bathrooms_clean'].isna().mean()*100:.1f}%)", '-- vs.',
      listings['bathrooms'].isna().sum(), 'missing in the raw numeric bathrooms field')

Missing after parsing bathrooms_text: 28 (0.2%) -- vs. 8573 missing in the raw numeric bathrooms field


### 3.5 Imputing bedrooms, beds, and bathrooms

Rather than a single global median (which would ignore that a studio and a 4-bedroom house are systematically different), we impute using the median **within each `property_type`** (e.g., "Entire condo", "Private room in home"), falling back to the overall median for any property type that itself has no non-missing value for that field.

In [8]:
def impute_grouped_median(df, col, group_col='property_type'):
    group_median = df.groupby(group_col)[col].transform('median')
    overall_median = df[col].median()
    return df[col].fillna(group_median).fillna(overall_median)

for col in ['bedrooms', 'beds', 'bathrooms_clean']:
    before = listings[col].isna().sum()
    listings[col + '_imp'] = impute_grouped_median(listings, col)
    after = listings[col + '_imp'].isna().sum()
    print(f'{col}: {before} missing -> {after} missing after property-type-median imputation')

bedrooms: 4080 missing -> 0 missing after property-type-median imputation
beds: 8313 missing -> 0 missing after property-type-median imputation
bathrooms_clean: 28 missing -> 0 missing after property-type-median imputation


### 3.6 Review scores: imputing with a missingness flag

For `review_scores_rating`, we impute the median rating but **also keep a `review_scores_missing` flag**. A listing with no reviews yet is systematically different from one with a mediocre rating, and collapsing the two into the same imputed value would quietly bias the price relationship if that distinction matters.

In [9]:
listings['review_scores_missing'] = listings['review_scores_rating'].isna().astype(int)
listings['review_scores_rating_imp'] = listings['review_scores_rating'].fillna(
    listings['review_scores_rating'].median()
)
print('review_scores_missing rate:', listings['review_scores_missing'].mean().round(3))
print('Median rating used for imputation:', listings['review_scores_rating'].median())

review_scores_missing rate: 0.179
Median rating used for imputation: 4.9


### 3.7 Dropping rows with missing price or minimum_nights

We cannot model a missing dependent variable, so the 861 listings (5.4%) with no price are dropped. `minimum_nights` is missing for a handful of listings (5) — small enough to drop rather than impute.

In [10]:
before = len(listings)
listings = listings[listings['price'].notna()].copy()
listings = listings[listings['minimum_nights'].notna()].copy()
after = len(listings)
print(f'Dropped {before - after} rows ({(before-after)/before*100:.1f}%). Modeling dataset: {after} listings.')

Dropped 866 rows (5.5%). Modeling dataset: 15003 listings.


### 3.8 License → `licensed` dummy variable

Toronto's Municipal Code Chapter 547 requires short-term rental operators to register with the City and display a registration number (format `STR-YYYY-XXXXXX`) on their listing. We convert the raw `license` field into a binary `licensed` indicator — this is the variable at the heart of our Chow test.

In [11]:
listings['licensed'] = listings['license'].notna().astype(int)

print('Overall licensed rate:', listings['licensed'].mean().round(3))
print()
print('Licensed rate by room type:')
print(listings.groupby('room_type')['licensed'].mean().round(3))
print()
print('Sample license values:')
print(listings['license'].dropna().head(5).tolist())

Overall licensed rate: 0.661

Licensed rate by room type:
room_type
Entire home/apt   0.675
Hotel room        0.070
Private room      0.636
Shared room       0.529
Name: licensed, dtype: float64

Sample license values:
['STR-2009-FXRRPD', 'STR-2604-GQRHPT', 'STR-2408-JCKSBZ', 'STR-2010-FPPDPC', 'STR-2012-HLWPHK']


### 3.9 Host and room-type flags, and grouping the 140 neighbourhoods

`host_is_superhost` arrives as `'t'`/`'f'` text — we convert to 0/1. For neighbourhood, 140 raw categories is too many to use individually as dummies without many of them being estimated on a handful of listings — we keep the 14 most common neighbourhoods as their own category and group everything else into `'Other'`.

In [12]:
listings['host_is_superhost_bin'] = (listings['host_is_superhost'] == 't').astype(int)

TOP_N_NEIGHBOURHOODS = 14
top_neighbourhoods = listings['neighbourhood_cleansed'].value_counts().head(TOP_N_NEIGHBOURHOODS).index.tolist()
listings['neighbourhood_grp'] = listings['neighbourhood_cleansed'].where(
    listings['neighbourhood_cleansed'].isin(top_neighbourhoods), 'Other'
)

print(f'Neighbourhood categories after grouping: {listings["neighbourhood_grp"].nunique()} '
      f'(top {TOP_N_NEIGHBOURHOODS} + Other)')
listings['neighbourhood_grp'].value_counts()

Neighbourhood categories after grouping: 15 (top 14 + Other)


neighbourhood_grp
Other                                  7621
Waterfront Communities-The Island      2641
Niagara                                 618
Annex                                   483
Church-Yonge Corridor                   471
Moss Park                               466
Dovercourt-Wallace Emerson-Junction     374
Kensington-Chinatown                    370
Bay Street Corridor                     359
Trinity-Bellwoods                       353
South Riverdale                         277
Willowdale East                         274
Little Portugal                         271
Palmerston-Little Italy                 226
York University Heights                 199
Name: count, dtype: int64

### 3.10 Engineered feature: distance to downtown

We compute each listing's distance to downtown Toronto (CN Tower, 43.6426°N, 79.3871°W) from its latitude/longitude using the haversine formula. We'll use this for exploratory mapping; whether it earns a place in the regression itself (alongside neighbourhood dummies, which already capture location).

In [13]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth's radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

CN_TOWER = (43.6426, -79.3871)
listings['dist_to_downtown_km'] = haversine_km(
    listings['latitude'], listings['longitude'], *CN_TOWER
)
listings['dist_to_downtown_km'].describe().round(2)

count   15,003.000
mean         6.870
std          6.060
min          0.070
25%          1.840
50%          4.650
75%         11.330
max         26.810
Name: dist_to_downtown_km, dtype: float64

### Recap: the cleaned dataset

In [14]:
print(f'Final modeling dataset: {listings.shape[0]:,} listings, {listings.shape[1]} columns')
print()
summary_cols = ['price', 'accommodates', 'bedrooms_imp', 'beds_imp', 'bathrooms_clean_imp',
                 'minimum_nights', 'dist_to_downtown_km']
listings[summary_cols].describe().round(2)

Final modeling dataset: 15,003 listings, 98 columns



,price,accommodates,bedrooms_imp,beds_imp,bathrooms_clean_imp,minimum_nights,dist_to_downtown_km
count,"15,003.000","15,003.000","15,003.000","15,003.000","15,003.000","15,003.000","15,003.000"
mean,275.050,3.450,1.550,1.970,1.310,16.060,6.870
std,359.190,2.150,0.890,1.030,0.650,21.130,6.060
min,6.080,1.000,0.000,1.000,0.000,1.000,0.070
25%,100.560,2.000,1.000,1.000,1.000,1.000,1.840
50%,193.000,3.000,1.000,2.000,1.000,3.000,4.650
75%,357.000,4.000,2.000,2.000,1.500,28.000,11.330
max,"11,042.360",16.000,10.000,13.000,9.000,365.000,26.810


## 4. Exploratory Data Analysis

### 4.1 Price distribution: raw vs. log

We already saw that mean CA275 sits well above median CA193. Plotting the distribution directly shows why: a long right tail of premium and luxury listings. Plotting `log(price)` alongside it shows the transformation we'll use later in Section 8 to fix this.

In [15]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Nightly Price ($)', 'log(Nightly Price)'))

fig.add_trace(go.Histogram(x=listings['price'], nbinsx=60, marker_color=COLOR_MAIN,
                            name='Price'), row=1, col=1)
fig.add_trace(go.Histogram(x=np.log(listings['price']), nbinsx=60, marker_color=COLOR_ACCENT,
                            name='log(Price)'), row=1, col=2)

fig.update_layout(template=PLOTLY_TEMPLATE, height=420, showlegend=False,
                   title_text='Distribution of Toronto Airbnb Nightly Price')
fig.update_xaxes(title_text='Price ($)', row=1, col=1)
fig.update_xaxes(title_text='log(Price)', row=1, col=2)
fig.update_yaxes(title_text='Number of listings', row=1, col=1)
fig.show()

**Insights:** the raw price histogram is heavily right-skewed with a long thin tail out past CA5,000 — a small number of ultra-premium listings. The log-price histogram is far closer to symmetric and bell-shaped, which is exactly the shape OLS assumes the *dependent variable's conditional distribution* should have.

### 4.2 Price by room type

In [16]:
fig = px.box(listings, x='room_type', y='price', color='room_type',
             template=PLOTLY_TEMPLATE, points=False,
             category_orders={'room_type': listings['room_type'].value_counts().index.tolist()},
             title='Nightly Price by Room Type')
fig.update_layout(showlegend=False, yaxis_title='Price ($)', xaxis_title='Room Type')
fig.update_yaxes(range=[0, 800])  # zoom past the extreme outliers for readability; box still reflects full data
fig.show()

### 4.3 Price by neighbourhood (top 14 + Other)

In [17]:
nbhd_order = (listings.groupby('neighbourhood_grp')['price'].median()
              .sort_values(ascending=False).index.tolist())

fig = px.box(listings, x='neighbourhood_grp', y='price', color='neighbourhood_grp',
             template=PLOTLY_TEMPLATE, points=False,
             category_orders={'neighbourhood_grp': nbhd_order},
             title='Nightly Price by Neighbourhood (ordered by median price)')
fig.update_layout(showlegend=False, yaxis_title='Price ($)', xaxis_title='Neighbourhood',
                   xaxis_tickangle=-40, height=550)
fig.update_yaxes(range=[0, 800])
fig.show()

### 4.4 Interactive map of listings

Every dot is one listing, colored by price and sized by how many guests it accommodates. Zooming in on the visualization is possible.

In [18]:
map_sample = listings.sample(n=min(6000, len(listings)), random_state=42)  # keep the map responsive

fig = px.scatter_mapbox(
    map_sample, lat='latitude', lon='longitude', color='price', size='accommodates',
    color_continuous_scale='Viridis', range_color=[50, 500], size_max=12, zoom=10.2,
    hover_data={'price': True, 'room_type': True, 'neighbourhood_cleansed': True,
                'latitude': False, 'longitude': False},
    center={'lat': 43.6426, 'lon': -79.3871},
    mapbox_style='open-street-map', template=PLOTLY_TEMPLATE,
    title='Toronto Airbnb Listings by Price'
)
fig.update_layout(height=600, margin={'r': 0, 't': 40, 'l': 0, 'b': 0})
fig.show()

### 4.5 Price vs. distance to downtown

Does price fall off as listings get further from the downtown core? We are using trendlines (fit by OLS), one line per room type to read against the noise of individual listings.

In [19]:
plot_df = listings[listings['price'] < 1000]  # trim extreme prices for a readable trend view

fig = px.scatter(plot_df, x='dist_to_downtown_km', y='price', color='room_type',
                  opacity=0.25, trendline='ols', trendline_scope='trace',
                  template=PLOTLY_TEMPLATE,
                  labels={'dist_to_downtown_km': 'Distance to Downtown (km)', 'price': 'Price ($)'},
                  title='Price vs. Distance to Downtown, by Room Type')
fig.update_layout(height=500)
fig.show()

### 4.6 Licensed vs. unlicensed pricing

Do listings that display a Chapter 547 registration number price differently than those that don't?

In [20]:
listings['licensed_label'] = listings['licensed'].map({1: 'Licensed', 0: 'Unlicensed'})

fig = px.box(listings, x='licensed_label', y='price', color='licensed_label',
             template=PLOTLY_TEMPLATE, points=False,
             color_discrete_map={'Licensed': COLOR_MAIN, 'Unlicensed': COLOR_ACCENT},
             title='Nightly Price: Licensed vs. Unlicensed Listings')
fig.update_layout(showlegend=False, yaxis_title='Price ($)', xaxis_title='')
fig.update_yaxes(range=[0, 800])
fig.show()

print(listings.groupby('licensed_label')['price'].agg(['mean', 'median', 'count']).round(2))

                  mean  median  count
licensed_label                       
Licensed       331.400 276.520   9920
Unlicensed     165.070 106.740   5083


**Insight:** licensed listings have a noticeably higher mean *and* median price than unlicensed ones. That's a big enough gap to be worth testing formally rather than reading into directly — plenty of other factors (room type, size, location) could be driving it.

### 4.7 Correlation heatmap of numeric predictors

In [21]:
corr_vars = ['price', 'accommodates', 'bedrooms_imp', 'beds_imp', 'bathrooms_clean_imp',
             'minimum_nights', 'review_scores_rating_imp', 'hosts_time_as_host_years',
             'dist_to_downtown_km']
corr_matrix = listings[corr_vars].corr().round(2)

fig = px.imshow(corr_matrix, text_auto=True, color_continuous_scale='RdBu', zmin=-1, zmax=1,
                 template=PLOTLY_TEMPLATE, title='Correlation Matrix: Numeric Predictors')
fig.update_layout(height=550)
fig.show()

**Insights:** `accommodates`, `bedrooms`, `beds`, and `bathrooms` are all positively correlated with each other (0.6–0.9 range) — the "size" cluster we flagged in our project proposal as a potential collinearity concern. We check this properly with VIF in Section 6.1 before deciding whether to act on it.

### 4.8 Choropleth: average price by neighbourhood

Using the neighbourhood boundary polygons from Inside Airbnb, we can map average price directly onto the shape of each of Toronto's 140 neighbourhoods rather than approximating with dots.

In [22]:
nbhd_avg_price = listings.groupby('neighbourhood_cleansed')['price'].mean().reset_index()

fig = px.choropleth_mapbox(
    nbhd_avg_price, geojson=neighbourhoods_geo, locations='neighbourhood_cleansed',
    featureidkey='properties.neighbourhood', color='price',
    color_continuous_scale='Viridis', range_color=[100, 400],
    mapbox_style='open-street-map', zoom=9.7,
    center={'lat': 43.6426, 'lon': -79.3871}, opacity=0.7,
    labels={'price': 'Avg. Price ($)'}, template=PLOTLY_TEMPLATE,
    title='Average Nightly Price by Toronto Neighbourhood'
)
fig.update_layout(height=600, margin={'r': 0, 't': 40, 'l': 0, 'b': 0})
fig.show()

## 5. Feature Engineering for Modeling

We now build the design matrix: dummy variables for `room_type` and `neighbourhood_grp`, then assemble the full modeling table.

In [23]:
room_dummies = pd.get_dummies(listings['room_type'], prefix='room', drop_first=True).astype(int)
nbhd_dummies = pd.get_dummies(listings['neighbourhood_grp'], prefix='nbhd', drop_first=True).astype(int)

print('Base case (omitted) room type:',
      set(listings['room_type'].unique()) - set(c.replace('room_', '') for c in room_dummies.columns))
print('Base case (omitted) neighbourhood:',
      set(listings['neighbourhood_grp'].unique()) - set(c.replace('nbhd_', '') for c in nbhd_dummies.columns))

model_df = pd.concat([
    listings[['price', 'accommodates', 'bedrooms_imp', 'beds_imp', 'bathrooms_clean_imp',
              'host_is_superhost_bin', 'hosts_time_as_host_years', 'review_scores_rating_imp',
              'minimum_nights', 'licensed']].reset_index(drop=True),
    room_dummies.reset_index(drop=True),
    nbhd_dummies.reset_index(drop=True),
], axis=1)

model_df['log_price'] = np.log(model_df['price'])

print(f'\nModeling table: {model_df.shape[0]:,} rows, {model_df.shape[1]} columns')
model_df.head()

Base case (omitted) room type: {'Entire home/apt'}
Base case (omitted) neighbourhood: {'Annex'}

Modeling table: 15,003 rows, 28 columns


,price,accommodates,bedrooms_imp,beds_imp,bathrooms_clean_imp,host_is_superhost_bin,hosts_time_as_host_years,review_scores_rating_imp,minimum_nights,licensed,room_Hotel room,room_Private room,room_Shared room,nbhd_Bay Street Corridor,nbhd_Church-Yonge Corridor,nbhd_Dovercourt-Wallace Emerson-Junction,nbhd_Kensington-Chinatown,nbhd_Little Portugal,nbhd_Moss Park,nbhd_Niagara,nbhd_Other,nbhd_Palmerston-Little Italy,nbhd_South Riverdale,nbhd_Trinity-Bellwoods,nbhd_Waterfront Communities-The Island,nbhd_Willowdale East,nbhd_York University Heights,log_price
0,99.850,2,1.000,2.000,1.000,1,15,4.190,28.000,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,4.604
1,76.700,2,1.000,1.000,1.000,1,15,4.000,30.000,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,4.340
2,107.410,3,1.000,2.000,1.000,1,15,4.170,28.000,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,4.677
3,71.860,1,1.000,1.000,1.000,1,15,4.400,28.000,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,4.275
4,112.000,3,1.000,2.000,1.000,1,15,4.630,28.000,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,4.718


### Train/test split

We hold out 20% of the data for out-of-sample validation in Section 12, so that the diagnostics and hypothesis tests in Sections 6–10 are run on a training set and the final accuracy check is genuinely out-of-sample.

In [24]:
from numpy.random import default_rng

rng = default_rng(42)
n_total = len(model_df)
shuffled_idx = rng.permutation(n_total)
test_size = int(0.2 * n_total)

test_idx = shuffled_idx[:test_size]
train_idx = shuffled_idx[test_size:]

train_df = model_df.iloc[train_idx].reset_index(drop=True)
test_df = model_df.iloc[test_idx].reset_index(drop=True)

print(f'Train: {len(train_df):,} listings   Test: {len(test_df):,} listings')

# x_cols is the full predictor list used throughout the rest of the notebook
x_cols = (['accommodates', 'bedrooms_imp', 'beds_imp', 'bathrooms_clean_imp']
          + list(room_dummies.columns) + list(nbhd_dummies.columns)
          + ['host_is_superhost_bin', 'hosts_time_as_host_years', 'review_scores_rating_imp',
             'minimum_nights', 'licensed'])
print(f'{len(x_cols)} predictors:')
print(x_cols)

Train: 12,003 listings   Test: 3,000 listings
26 predictors:
['accommodates', 'bedrooms_imp', 'beds_imp', 'bathrooms_clean_imp', 'room_Hotel room', 'room_Private room', 'room_Shared room', 'nbhd_Bay Street Corridor', 'nbhd_Church-Yonge Corridor', 'nbhd_Dovercourt-Wallace Emerson-Junction', 'nbhd_Kensington-Chinatown', 'nbhd_Little Portugal', 'nbhd_Moss Park', 'nbhd_Niagara', 'nbhd_Other', 'nbhd_Palmerston-Little Italy', 'nbhd_South Riverdale', 'nbhd_Trinity-Bellwoods', 'nbhd_Waterfront Communities-The Island', 'nbhd_Willowdale East', 'nbhd_York University Heights', 'host_is_superhost_bin', 'hosts_time_as_host_years', 'review_scores_rating_imp', 'minimum_nights', 'licensed']


## 6. Model 1: Baseline Multiple Regression

Models for assessment: (1) joint F-test, (2) individual t-tests, (3) diagnostic assumptions, (4) R²

Before that we shall check collinearity among the size variables.

### 6.1 Collinearity check: VIF

Section 4.7's correlation heatmap showed `accommodates`, `bedrooms`, `beds`, and `bathrooms` correlated with each other in the 0.6–0.9 range — worth checking properly before deciding whether any of them need to be dropped or combined. Raw pairwise correlation isn't the right diagnostic for this (two variables can be highly correlated with each other yet still both contribute cleanly once the rest of the model is accounted for) — **Variance Inflation Factor (VIF)** is.

In [25]:
def compute_vif(df, cols):
    vifs = {}
    for col in cols:
        others = [c for c in cols if c != col]
        X_others = sm.add_constant(df[others])
        r2 = sm.OLS(df[col], X_others).fit().rsquared
        vifs[col] = 1 / (1 - r2) if r2 < 1 else np.inf
    return pd.Series(vifs, name='VIF').sort_values(ascending=False)

size_vars = ['accommodates', 'bedrooms_imp', 'beds_imp', 'bathrooms_clean_imp']
compute_vif(train_df, size_vars).round(2)

accommodates          4.000
bedrooms_imp          3.940
beds_imp              2.570
bathrooms_clean_imp   1.990
Name: VIF, dtype: float64

**Insights:** every VIF here is comfortably below 5, let alone the stricter 10. Despite the pairwise correlations of up to 0.89, the multivariate collinearity is mild — each of the four size variables still carries enough independent information to stay in the model. We keep all four rather than mechanically dropping one.

### 6.2 Fitting the baseline model

$$Price = \beta_0 + \beta_1 Accommodates + \beta_2 Bedrooms + \beta_3 Beds + \beta_4 Bathrooms + \beta_5 RoomType_j + \beta_6 Neighbourhood_k + \beta_7 Superhost + \beta_8 HostTenure + \beta_9 ReviewScore + \beta_{10} MinNights + \beta_{11} Licensed + \varepsilon$$

In [26]:
X_train = sm.add_constant(train_df[x_cols])
y_train = train_df['price']

model1 = sm.OLS(y_train, X_train).fit()
print(model1.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.303
Model:                            OLS   Adj. R-squared:                  0.301
Method:                 Least Squares   F-statistic:                     199.8
Date:                Tue, 04 Aug 2026   Prob (F-statistic):               0.00
Time:                        21:02:37   Log-Likelihood:                -85062.
No. Observations:               12003   AIC:                         1.702e+05
Df Residuals:                   11976   BIC:                         1.704e+05
Df Model:                          26                                         
Covariance Type:            nonrobust                                         
                                               coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------

### Interpreting Model 1

**Step 1 — does the model have predictive power?** The F-statistic's p-value is effectively 0, so yes: we can reject the null that all coefficients are jointly zero.

**Step 2 — do the individual variables belong?** Most do — `accommodates`, `bedrooms`, `beds`, `bathrooms`, room type, `minimum_nights`, `superhost`, and `licensed` are all significant at conventional levels. A few individual neighbourhood dummies (e.g. Niagara, Palmerston-Little Italy) are not significant on their own — expected when a base case is a specific well-known area (the Annex) and some other areas simply don't differ much in price from it. We test the neighbourhood block *jointly* in Section 10.1 rather than reading too much into any single one of these t-tests.

**Notable coefficients:**
- Each additional bathroom is associated with the largest single price jump among the size variables.
- `Private room` is associated with a substantially lower price than the omitted base case, `Entire home/apt`.
- `licensed` carries a large, positive, significant coefficient — consistent with what the raw box plot in Section 4.6 showed, and the reason this needs the formal Chow test in Section 10.2 rather than being read directly off this one coefficient (a positive `licensed` coefficient here says licensed listings are priced higher *holding other characteristics fixed* — it doesn't yet tell us whether the whole *pricing relationship* differs between the two groups).

**Step 4 — is R² sufficient?** We'll hold off judging this until Section 8, where the log-price model raises it substantially — a strong sign the raw-price model is a while away from being correctly specified.

## 7. Diagnostics

We check the full-model VIF, then build the four standard diagnostic plots interactively, then run a formal Breusch-Pagan test, then look at Cook's Distance for influential outliers.

### 7.1 Full-model VIF

In [27]:
compute_vif(train_df, x_cols).round(2)

nbhd_Other                                 8.390
nbhd_Waterfront Communities-The Island     5.320
accommodates                               4.590
bedrooms_imp                               4.230
beds_imp                                   3.010
nbhd_Niagara                               2.150
bathrooms_clean_imp                        2.040
nbhd_Church-Yonge Corridor                 1.940
nbhd_Moss Park                             1.900
nbhd_Kensington-Chinatown                  1.780
nbhd_Bay Street Corridor                   1.730
room_Private room                          1.730
nbhd_Dovercourt-Wallace Emerson-Junction   1.710
nbhd_Trinity-Bellwoods                     1.680
nbhd_Willowdale East                       1.540
licensed                                   1.540
nbhd_South Riverdale                       1.530
nbhd_Little Portugal                       1.510
minimum_nights                             1.480
nbhd_Palmerston-Little Italy               1.460
nbhd_York University

All predictors remain under 10; the highest (the `Other` neighbourhood bucket and Waterfront Communities, both in the 5–8 range) is expected since they're large, catch-all/dominant categories, not a sign of a broken model.

### 7.2 Residual diagnostics

In [28]:
influence1 = model1.get_influence()
leverage1 = influence1.hat_matrix_diag
std_resid1 = influence1.resid_studentized_internal
cooks_d1 = influence1.cooks_distance[0]

fig = make_subplots(rows=2, cols=2, subplot_titles=(
    'Residuals vs. Fitted', 'Normal Q-Q', 'Scale-Location', 'Residuals vs. Leverage'
))

# 1. Residuals vs Fitted
fig.add_trace(go.Scatter(x=model1.fittedvalues, y=model1.resid, mode='markers',
                          marker=dict(color=COLOR_MAIN, size=4, opacity=0.4)), row=1, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='red', row=1, col=1)

# 2. Normal Q-Q
qq = stats.probplot(std_resid1, dist='norm')
fig.add_trace(go.Scatter(x=qq[0][0], y=qq[0][1], mode='markers',
                          marker=dict(color=COLOR_MAIN, size=4, opacity=0.4)), row=1, col=2)
qq_line_x = np.array([qq[0][0].min(), qq[0][0].max()])
fig.add_trace(go.Scatter(x=qq_line_x, y=qq[1][0] * qq_line_x + qq[1][1], mode='lines',
                          line=dict(color='red', dash='dash')), row=1, col=2)

# 3. Scale-Location
sqrt_std_resid1 = np.sqrt(np.abs(std_resid1))
fig.add_trace(go.Scatter(x=model1.fittedvalues, y=sqrt_std_resid1, mode='markers',
                          marker=dict(color=COLOR_MAIN, size=4, opacity=0.4)), row=2, col=1)

# 4. Residuals vs Leverage
fig.add_trace(go.Scatter(x=leverage1, y=std_resid1, mode='markers',
                          marker=dict(color=COLOR_MAIN, size=4 + 40 * cooks_d1, opacity=0.4)),
              row=2, col=2)
fig.add_hline(y=0, line_dash='dash', line_color='red', row=2, col=2)

fig.update_xaxes(title_text='Fitted values', row=1, col=1)
fig.update_yaxes(title_text='Residuals', row=1, col=1)
fig.update_xaxes(title_text='Theoretical quantiles', row=1, col=2)
fig.update_yaxes(title_text='Sample quantiles', row=1, col=2)
fig.update_xaxes(title_text='Fitted values', row=2, col=1)
fig.update_yaxes(title_text='sqrt(|standardized residuals|)', row=2, col=1)
fig.update_xaxes(title_text='Leverage', row=2, col=2)
fig.update_yaxes(title_text='Standardized residuals', row=2, col=2)

fig.update_layout(template=PLOTLY_TEMPLATE, height=750, showlegend=False,
                   title_text='Model 1 Diagnostic Plots (marker size in panel 4 = Cook\'s Distance)')
fig.show()

**Insights:** the residuals-vs-fitted panel fans out — the spread of residuals visibly widens as fitted price increases — which is the visual signature of **heteroskedasticity**. The Q-Q plot's upper tail bows away from the reference line, showing the residuals are more heavy-tailed than a normal distribution (driven by the same handful of ultra-premium listings).

### 7.3 Breusch-Pagan test for heteroskedasticity

$H_0$: the error variance is constant (homoskedastic). We regress the squared residuals on the model's predictors and test whether that auxiliary regression has any explanatory power.

In [29]:
bp_stat, bp_pvalue, bp_fstat, bp_fpvalue = het_breuschpagan(model1.resid, model1.model.exog)
labels = ['LM Statistic', 'LM-Test p-value', 'F-Statistic', 'F-Test p-value']
print(dict(zip(labels, [bp_stat, bp_pvalue, bp_fstat, bp_fpvalue])))

{'LM Statistic': np.float64(59.28658788288794), 'LM-Test p-value': np.float64(0.0002089802605175781), 'F-Statistic': np.float64(2.2864174263010764), 'F-Test p-value': np.float64(0.00020436669135016534)}


**Insight:** the p-value is far below 0.05 — we reject the null of homoskedasticity. Combined with the visual fan pattern above and the fact that the raw price distribution is strongly right-skewed (Section 4.1), this points squarely at a **log transformation** of the dependent variable, which we implement in Section 8.

### 7.4 Cook's Distance: influential observations

In [30]:
n_train = len(train_df)
cooks_threshold = 4 / n_train

fig = go.Figure()
fig.add_trace(go.Bar(x=list(range(n_train)), y=cooks_d1, marker_color=COLOR_MAIN,
                      name="Cook's D"))
fig.add_hline(y=cooks_threshold, line_dash='dash', line_color='red',
              annotation_text=f'Threshold = 4/n = {cooks_threshold:.5f}')
fig.update_layout(template=PLOTLY_TEMPLATE, height=420,
                   title="Cook's Distance by Observation (Model 1)",
                   xaxis_title='Observation index (training set)', yaxis_title="Cook's Distance")
fig.show()

flagged1 = cooks_d1 > cooks_threshold
print(f'Threshold (4/n): {cooks_threshold:.6f}')
print(f"Largest Cook's Distance: {cooks_d1.max():.4f}")
print(f'Observations exceeding threshold: {flagged1.sum()} ({flagged1.mean()*100:.1f}%)')

top5_idx = np.argsort(cooks_d1)[::-1][:5]
print('\nTop 5 most influential listings:')
train_df.iloc[top5_idx][['price', 'accommodates', 'bedrooms_imp']].assign(cooks_d=cooks_d1[top5_idx])

Threshold (4/n): 0.000333
Largest Cook's Distance: 0.1229
Observations exceeding threshold: 237 (2.0%)

Top 5 most influential listings:


,price,accommodates,bedrooms_imp,cooks_d
7382,"7,753.550",15,5.000,0.123
8886,"5,294.860",10,7.000,0.101
1079,"7,000.000",2,1.000,0.064
8370,"6,196.920",2,1.000,0.049
3742,"6,132.800",4,2.000,0.044


**Insights:** the most influential observations are, unsurprisingly, among the highest-priced listings in the training set — their Cook's Distance is an order of magnitude above the rest. This is to note that *which* specific listings show up here depends on the random train/test split in Section 5: an extreme-price listing only appears in this diagnostic if it happened to land in the training set rather than the held-out test set. About 1–2% of observations sit above the conventional 4/n threshold, which on a dataset this size is not alarming on its own, but we revisit this properly with a robustness check once we've moved to the better-specified log-price model (Section 9).

## 8. Refined Model: Log-Price Transformation

Three independent signals all pointed the same direction: the raw price distribution is right-skewed (4.1), the residual plots fan out (7.2), and the Breusch-Pagan test formally rejects homoskedasticity (7.3). We re-estimate with $\log(Price)$ as the dependent variable.

In [31]:
y_train_log = train_df['log_price']

model1_log = sm.OLS(y_train_log, X_train).fit()
print(model1_log.summary())

                            OLS Regression Results                            
Dep. Variable:              log_price   R-squared:                       0.730
Model:                            OLS   Adj. R-squared:                  0.729
Method:                 Least Squares   F-statistic:                     1244.
Date:                Tue, 04 Aug 2026   Prob (F-statistic):               0.00
Time:                        21:02:37   Log-Likelihood:                -7453.8
No. Observations:               12003   AIC:                         1.496e+04
Df Residuals:                   11976   BIC:                         1.516e+04
Df Model:                          26                                         
Covariance Type:            nonrobust                                         
                                               coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------

In [32]:
print(f'Linear model  R-squared: {model1.rsquared:.4f}')
print(f'Log model     R-squared: {model1_log.rsquared:.4f}')
print()
print(f'Linear model  AIC: {model1.aic:.1f}')
print(f'Log model     AIC: {model1_log.aic:.1f}')

Linear model  R-squared: 0.3025
Log model     R-squared: 0.7299

Linear model  AIC: 170178.3
Log model     AIC: 14961.7


**Insight:** R² roughly doubles — from the 0.3 range to the 0.7 range — and AIC drops sharply — both say the log-price specification fits the data dramatically better. This size of an improvement is a strong sign the raw-price model in Section 6 was misspecified (not just heteroskedastic), and that price affects — and is affected by — these predictors multiplicatively, not additively. This also lets us interpret each coefficient as an approximate **percentage** effect on price: $(\,e^{\beta} - 1) \times 100\%$.

In [33]:
coef_table = model1_log.params.to_frame('coef')
coef_table['pct_effect_on_price'] = (np.exp(coef_table['coef']) - 1) * 100
coef_table['p_value'] = model1_log.pvalues
coef_table.round(4)

,coef,pct_effect_on_price,p_value
const,4.590,"9,744.117",0.000
accommodates,0.101,10.658,0.000
bedrooms_imp,0.122,12.992,0.000
beds_imp,-0.058,-5.606,0.000
bathrooms_clean_imp,0.131,14.001,0.000
room_Hotel room,0.319,37.599,0.000
room_Private room,-0.581,-44.074,0.000
room_Shared room,-0.698,-50.257,0.000
nbhd_Bay Street Corridor,0.047,4.826,0.175
nbhd_Church-Yonge Corridor,-0.042,-4.098,0.196


**Note:** each additional bathroom is associated with roughly a 14% higher price, holding everything else fixed; switching from a Private Room to an Entire Home/Apt is associated with a much larger jump; each additional minimum-night requirement is associated with a small price *discount* (hosts requiring longer minimum stays tend to price a bit lower per night); and `licensed` carries a large, positive, significant effect on price.

### Why does `beds_imp` flip sign?

In the coefficient table above, `beds_imp` carries a negative sign (-5.6%) even though Section 4.7's correlation heatmap showed it positively correlated with price (0.6-0.9 range alongside the other size variables). Before treating a negative beds coefficient as a red flag, it's worth checking whether this is a genuine partial effect or a multicollinearity artifact — `compute_vif` and `size_vars` are already defined in Section 6.1, so we reuse them here.

In [34]:
# Bivariate vs. full-model coefficient on beds_imp
X_beds_only = sm.add_constant(train_df[['beds_imp']])
model_beds_only = sm.OLS(y_train_log, X_beds_only).fit()

beds_bivariate = model_beds_only.params['beds_imp']
beds_full = model1_log.params['beds_imp']
beds_vif = compute_vif(train_df, size_vars)['beds_imp']

print(f"beds_imp coefficient, alone:      {beds_bivariate:+.4f}  ({(np.exp(beds_bivariate)-1)*100:+.1f}% price effect)")
print(f"beds_imp coefficient, full model: {beds_full:+.4f}  ({(np.exp(beds_full)-1)*100:+.1f}% price effect)")
print(f"beds_imp VIF (from Section 6.1):  {beds_vif:.2f}")

beds_imp coefficient, alone:      +0.4175  (+51.8% price effect)
beds_imp coefficient, full model: -0.0577  (-5.6% price effect)
beds_imp VIF (from Section 6.1):  2.57


In [35]:
fig = go.Figure()
fig.add_trace(go.Bar(
    x=['Alone (bivariate)', 'Full model (Section 8)'],
    y=[(np.exp(beds_bivariate)-1)*100, (np.exp(beds_full)-1)*100],
    marker_color=[COLOR_ACCENT, COLOR_MAIN],
    text=[f'{(np.exp(beds_bivariate)-1)*100:+.1f}%', f'{(np.exp(beds_full)-1)*100:+.1f}%'],
    textposition='outside',
))
fig.add_hline(y=0, line_color='black', line_width=1)
fig.update_layout(template=PLOTLY_TEMPLATE, height=380, showlegend=False,
                   title="beds_imp's Price Effect: Alone vs. Holding Capacity, Bedrooms, Bathrooms and Room Type Fixed",
                   yaxis_title='% effect on price')
fig.show()

**Insights:** on its own, more beds is associated with a *higher* price — consistent with the 0.6-0.9 pairwise correlations in Section 4.7. Once `accommodates`, `bedrooms`, `bathrooms`, and `room_type` are all held fixed, the coefficient flips negative. This isn't a collinearity artifact — `beds_imp`'s VIF (2.57) stays well under the concern threshold. It's a genuine partial effect: for two listings with the *same* guest capacity, bedroom count, and bathroom count, the one that packs in more beds is priced lower. That's consistent with extra beds signalling bunk-style or shared-sleeping configurations (e.g. sofa beds or extra singles squeezed into an existing room) rather than added comfort, once capacity is already accounted for by `accommodates`. We keep the variable rather than dropping it — the finding is substantively interesting on its own, and the low VIF confirms it isn't destabilizing the rest of the model.

### Does the log transform fix the heteroskedasticity?

In [36]:
bp_log = het_breuschpagan(model1_log.resid, model1_log.model.exog)
print(dict(zip(labels, bp_log)))
print()
print(f'Linear model  Breusch-Pagan LM = {bp_stat:.2f}, p = {bp_pvalue:.3g}')
print(f'Log model     Breusch-Pagan LM = {bp_log[0]:.2f}, p = {bp_log[1]:.3g}')

{'LM Statistic': np.float64(561.8408770902287), 'LM-Test p-value': np.float64(5.240283665571716e-102), 'F-Statistic': np.float64(22.619434701799985), 'F-Test p-value': np.float64(1.1111233063158975e-104)}

Linear model  Breusch-Pagan LM = 59.29, p = 0.000209
Log model     Breusch-Pagan LM = 561.84, p = 5.24e-102


**An honest, slightly counter-intuitive finding:** the Breusch-Pagan test *still* rejects homoskedasticity for the log model — in fact the LM statistic is larger, not smaller. This doesn't mean the log transform failed: it fixed the dominant problem (variance scaling with the *level* of price, driven by a few extreme listings) but some remaining heteroskedasticity tied to categorical segments (e.g., shared/hotel rooms have very few observations and more erratic pricing) is still detectable — and with 12,000 training observations, the Breusch-Pagan test has enough statistical power to detect even fairly small remaining patterns. The textbook remedy for exactly this situation is not to keep transforming the data, but to use **heteroskedasticity-consistent (robust) standard errors** for inference, which is what we do next.

In [37]:
model1_log_robust = sm.OLS(y_train_log, X_train).fit(cov_type='HC1')

compare_se = pd.DataFrame({
    'coef': model1_log.params,
    'se (ols)': model1_log.bse,
    'p (ols)': model1_log.pvalues,
    'se (robust HC1)': model1_log_robust.bse,
    'p (robust HC1)': model1_log_robust.pvalues,
})
compare_se.round(4)

,coef,se (ols),p (ols),se (robust HC1),p (robust HC1)
const,4.590,0.067,0.000,0.075,0.000
accommodates,0.101,0.004,0.000,0.005,0.000
bedrooms_imp,0.122,0.009,0.000,0.012,0.000
beds_imp,-0.058,0.007,0.000,0.008,0.000
bathrooms_clean_imp,0.131,0.009,0.000,0.011,0.000
room_Hotel room,0.319,0.078,0.000,0.055,0.000
room_Private room,-0.581,0.012,0.000,0.012,0.000
room_Shared room,-0.698,0.117,0.000,0.121,0.000
nbhd_Bay Street Corridor,0.047,0.035,0.175,0.036,0.194
nbhd_Church-Yonge Corridor,-0.042,0.032,0.196,0.034,0.218


**Insights:** the robust standard errors are close to the OLS ones and every variable that was significant under standard OLS inference remains significant under the heteroskedasticity-robust version — our substantive conclusions are not an artifact of the variance assumption being violated. From here on, **`model1_log_robust` is the model we treat as final** for inference purposes.

### Linking Price Residuals to Occupancy: Testing the Overpricing Hypothesis

The project proposal's Problem Statement raised two related pricing failures: hosts **underpricing** (leaving revenue on the table) and hosts **overpricing** (facing suppressed occupancy), noting the second pattern should be directly observable in the dataset's occupancy fields. `calendar.csv` is exactly that data. 8.1 million date-level availability rows we can collapse into a per-listing occupancy rate.

We test the overpricing side of that claim directly. A listing's Section 8 residual is, by construction, how much higher or lower it's priced than the model predicts for a listing with its characteristics. If overpricing genuinely suppresses bookings, listings with large *positive* residuals (priced above what their characteristics predict) should show *lower* occupancy than listings with negative residuals.

**Occupancy proxy:** `available == 'f'` on a given calendar day means that day is blocked, which is our best available proxy for "booked or otherwise unavailable" — Inside Airbnb doesn't publish confirmed bookings directly, and a host can also block a date for personal reasons unrelated to price. We revisit this caveat in the interpretation below.

In [38]:
occupancy = (
    calendar.groupby('listing_id')['available']
    .apply(lambda s: (s == 'f').mean())
    .rename('occupancy_rate')
    .reset_index()
)

print(f'Listings with calendar coverage: {occupancy.shape[0]:,}')
print(occupancy['occupancy_rate'].describe().round(3))

Listings with calendar coverage: 22,226
count   22,226.000
mean         0.492
std          0.357
min          0.000
25%          0.142
50%          0.493
75%          0.827
max          1.000
Name: occupancy_rate, dtype: float64


In [39]:
# train_df was built directly from listings, in the same row order (Section 5), so listings.iloc[train_idx] gives the listing id for every row in train_df, in the same order.
train_ids = listings.iloc[train_idx]['id'].reset_index(drop=True)

resid_occ = pd.DataFrame({
    'id': train_ids,
    'resid': model1_log.resid,
}).merge(occupancy, left_on='id', right_on='listing_id', how='left')

missing_occ = resid_occ['occupancy_rate'].isna().sum()
resid_occ = resid_occ.dropna(subset=['occupancy_rate'])
print(f'Training listings matched to calendar data: {len(resid_occ):,} '
      f'({missing_occ} of {len(train_df)} had no calendar rows and were dropped from this check only)')

corr = resid_occ['resid'].corr(resid_occ['occupancy_rate'])
print(f'\nCorrelation(price residual, occupancy rate): {corr:.4f}')

Training listings matched to calendar data: 11,643 (360 of 12003 had no calendar rows and were dropped from this check only)

Correlation(price residual, occupancy rate): -0.0690


In [40]:
resid_occ['price_group'] = pd.qcut(resid_occ['resid'], q=4,
    labels=['Most underpriced (Q1)', 'Q2', 'Q3', 'Most overpriced (Q4)'])

print(resid_occ.groupby('price_group', observed=True)['occupancy_rate'].agg(['mean', 'median', 'count']).round(3))

fig = px.box(resid_occ, x='price_group', y='occupancy_rate', color='price_group',
             template=PLOTLY_TEMPLATE, points=False,
             category_orders={'price_group': ['Most underpriced (Q1)', 'Q2', 'Q3', 'Most overpriced (Q4)']},
             color_discrete_sequence=[COLOR_MAIN, '#7A9CB5', '#D08A87', COLOR_ACCENT],
             title='Occupancy Rate by Price-Residual Quartile (Training Set)')
fig.update_layout(showlegend=False, yaxis_title='Occupancy Rate (share of days blocked)', xaxis_title='')
fig.show()

                       mean  median  count
price_group                               
Most underpriced (Q1) 0.424   0.422   2911
Q2                    0.414   0.384   2911
Q3                    0.410   0.370   2910
Most overpriced (Q4)  0.378   0.296   2911


**Insights:** the correlation is small (-0.07) but the sample is large enough (n = 11,643) that it's not noise. Listings priced above what their characteristics predict do see somewhat lower occupancy. The quartile breakdown makes the pattern easier to read than the raw correlation. Median occupancy drops from 42% for the most-underpriced quartile to 30% for the most-overpriced quartile, even though the group means barely move. The mean might be pulled around by listings that are rarely available for reasons that have nothing to do with price, like being blocked for the host's own use.

This supports the overpricing side of our Problem Statement, though weakly. Mispricing on the high side is associated with lower occupancy, not just lower revenue-per-booking. It's a modest effect next to the structural drivers in Section 8, room type and location move price far more than a host's pricing error moves occupancy, and it remains correlational. Occupancy also depends on factors we haven't modeled (photos, response time, seasonality), and `available == 'f'` conflates actual bookings with host-initiated blocks. We can use this as a directional signal for hosts, not a precise occupancy forecast.

## 9. Outlier Robustness Check

Section 7.4 flagged a handful of high-Cook's-D listings under the *linear* model. We repeat that check under the better-specified log model, and then ask the practical question the proposal committed to: do the results change materially if we exclude the most influential points?

In [41]:
influence_log = model1_log.get_influence()
cooks_d_log = influence_log.cooks_distance[0]
flagged_log = cooks_d_log > cooks_threshold

print(f"Max Cook's D (log model): {cooks_d_log.max():.4f}")
print(f'Flagged: {flagged_log.sum()} listings ({flagged_log.mean()*100:.1f}%)')

train_df_clean = train_df.loc[~flagged_log].reset_index(drop=True)
X_train_clean = sm.add_constant(train_df_clean[x_cols])
model1_log_clean = sm.OLS(train_df_clean['log_price'], X_train_clean).fit()

print(f'\nWith all training listings:        R2 = {model1_log.rsquared:.4f}, n = {len(train_df)}')
print(f'Excluding flagged influential pts: R2 = {model1_log_clean.rsquared:.4f}, n = {len(train_df_clean)}')

Max Cook's D (log model): 0.0492
Flagged: 623 listings (5.2%)

With all training listings:        R2 = 0.7299, n = 12003
Excluding flagged influential pts: R2 = 0.8049, n = 11380


In [42]:
coef_compare = pd.DataFrame({
    'coef (full)': model1_log.params,
    'coef (excl. influential)': model1_log_clean.params,
}).round(4)
coef_compare['pct_change'] = ((coef_compare['coef (excl. influential)'] - coef_compare['coef (full)'])
                                / coef_compare['coef (full)'].abs() * 100).round(1)
coef_compare.sort_values('pct_change', key=abs, ascending=False)

,coef (full),coef (excl. influential),pct_change
nbhd_Waterfront Communities-The Island,-0.012,0.007,155.400
nbhd_Bay Street Corridor,0.047,0.019,-59.200
nbhd_Niagara,0.032,0.048,49.100
nbhd_Church-Yonge Corridor,-0.042,-0.056,-33.000
review_scores_rating_imp,0.106,0.139,31.400
room_Hotel room,0.319,0.226,-29.300
minimum_nights,-0.015,-0.018,-22.100
bedrooms_imp,0.122,0.142,16.200
hosts_time_as_host_years,0.006,0.007,15.400
nbhd_Palmerston-Little Italy,-0.150,-0.173,-15.200


**Insights:** excluding the ~5% most influential listings improves fit further (R² rises) and the **core size and structural predictors — `accommodates`, `bedrooms`, `beds`, `bathrooms`, room type, and `licensed`** — stay stable in sign and within roughly 1–12% of their original magnitude. The largest percentage swings are concentrated in a few small-magnitude neighbourhood coefficients that were close to zero (and not statistically significant) to begin with, so a small absolute change looks like a large percentage change — not a sign the model is fragile. 

**The headline findings are not an artifact of a handful of extreme listings.** We keep the full sample (all 15,003 listings) for the hypothesis tests that follow, since dropping data should be a last resort, not a default.

## 10. Hypothesis Testing

Rejecting individual t-tests one at a time inflates the chance of a false positive; testing a whole block of coefficients jointly with a single F-test is the correct approach.

We refit the log-price model on the **full modeling dataset** (not just the training split) for these tests, since hypothesis testing about the underlying population relationship should use all the data we have.

In [43]:
X_full = sm.add_constant(model_df[x_cols])
y_full_log = model_df['log_price']
model_full_log = sm.OLS(y_full_log, X_full).fit()

print(f'Full-sample log model: R2 = {model_full_log.rsquared:.4f}, n = {model_full_log.nobs:.0f}')

Full-sample log model: R2 = 0.7288, n = 15003


### 10.1 Joint F-test: does location matter at all?

Several individual neighbourhood dummies in Section 8's coefficient table were not statistically significant on their own. Rather than reading anything into those individually, we test the **entire block of 14 neighbourhood dummies jointly**:

$$H_0: \beta_{nbhd,1} = \beta_{nbhd,2} = \dots = \beta_{nbhd,14} = 0 \quad vs. \quad H_1: \text{at least one } \beta_{nbhd,k} \neq 0$$

We use `wald_test` with an explicit restriction matrix (one row per neighbourhood dummy, a 1 in that dummy's column and 0s elsewhere) — this is the general, robust way to test a block of coefficients and sidesteps any ambiguity in typing out a hypothesis string by hand.

In [44]:
param_names = X_full.columns.tolist()
nbhd_cols = [c for c in x_cols if c.startswith('nbhd_')]

R_nbhd = np.zeros((len(nbhd_cols), len(param_names)))
for i, col in enumerate(nbhd_cols):
    R_nbhd[i, param_names.index(col)] = 1

wald_nbhd = model_full_log.wald_test(R_nbhd)
print(wald_nbhd)

<F test: F=array([[170.89837846]]), p=0.0, df_denom=1.5e+04, df_num=14>


**Insights:** the p-value is effectively 0 — we reject $H_0$ decisively. Location matters jointly, even though several *individual* neighbourhoods were statistically indistinguishable from the Annex (the base case) on their own.

### 10.2 Chow test: do licensed and unlicensed listings price differently?

Toronto's Municipal Code Chapter 547 requires short-term rental operators to register and display a licence number; a licensed, entire-unit registration is capped at 180 nights/year and must be the operator's principal residence — meaningfully different economics than an unregistered listing. We test whether the **entire pricing relationship** — not just the average level — differs between the two groups.

$$Price_{eqn} = \beta_0 + \sum_k \beta_k X_k + \gamma_0 Licensed + \sum_k \gamma_k (X_k \times Licensed) + \varepsilon$$
$$H_0: \gamma_0 = \gamma_1 = \dots = \gamma_K = 0 \quad \text{(one pooled pricing relationship for both groups)}$$
$$H_1: \text{at least one } \gamma \neq 0 \quad \text{(licensed and unlicensed listings price differently)}$$

In [45]:
interaction_df = model_df.copy()
base_cols = [c for c in x_cols if c != 'licensed']  # every predictor except licensed itself

interaction_cols = []
for col in base_cols:
    new_col = f'{col}_x_licensed'
    interaction_df[new_col] = interaction_df[col] * interaction_df['licensed']
    interaction_cols.append(new_col)

unrestricted_cols = x_cols + interaction_cols  # pooled predictors + licensed + all interactions
X_chow_unrestricted = sm.add_constant(interaction_df[unrestricted_cols])
model_chow_unrestricted = sm.OLS(interaction_df['log_price'], X_chow_unrestricted).fit()

print(f'Unrestricted model (fully interacted with Licensed): '
      f'R2 = {model_chow_unrestricted.rsquared:.4f}, {len(unrestricted_cols)} predictors')

Unrestricted model (fully interacted with Licensed): R2 = 0.7382, 51 predictors


In [46]:
chow_param_names = X_chow_unrestricted.columns.tolist()
chow_test_cols = ['licensed'] + interaction_cols  # licensed dummy + every interaction term

R_chow = np.zeros((len(chow_test_cols), len(chow_param_names)))
for i, col in enumerate(chow_test_cols):
    R_chow[i, chow_param_names.index(col)] = 1

chow_wald = model_chow_unrestricted.wald_test(R_chow)
print('=== Chow test: licensed vs. unlicensed pricing relationship ===')
print(chow_wald)
print(f'\nRestrictions tested: {len(chow_test_cols)}  (Licensed dummy + {len(interaction_cols)} interaction terms)')

=== Chow test: licensed vs. unlicensed pricing relationship ===
<F test: F=array([[62.068391]]), p=0.0, df_denom=1.5e+04, df_num=26>

Restrictions tested: 26  (Licensed dummy + 25 interaction terms)


In [47]:
print('Group sizes and raw price comparison:')
print(model_df.groupby('licensed')['price'].agg(['count', 'mean', 'median']).round(2)
      .rename(index={0: 'Unlicensed', 1: 'Licensed'}))

Group sizes and raw price comparison:
            count    mean  median
licensed                         
Unlicensed   5083 165.070 106.740
Licensed     9920 331.400 276.520


### Interpreting the Chow test

**We reject $H_0$ decisively (p ≈ 0).** The pricing relationship for licensed listings is *structurally different* from unlicensed listings — not just offset by a constant amount, but built on a different relationship between characteristics (size, room type, location, review score, etc.) and price.

The raw numbers underneath this are striking: licensed listings average **\$331/night** (median \$277) versus **\$165/night** (median \$107) for unlicensed listings — roughly double. A few candidate explanations worth stating explicitly (this test tells us the relationship differs, not *why*):

- Licensed, Chapter 547-compliant operators skew toward **entire-home** listings (higher price tier) more than unlicensed ones, who may lean toward lower-cost partial-unit/room rentals that face less regulatory scrutiny.
- Registered operators may be more established, professionally managed, or amenity-rich listings that would command a premium anyway, independent of licensing itself.
- Some unlicensed listings may be newer, informally operated, or priced defensively low precisely *because* they are operating outside the registration system.

**This is a correlational finding, not a causal one** — the Chow test tells us licensing status coincides with a structurally different pricing regime, not that becoming licensed *causes* a listing's price to change. For the City's policy discussion, it's still a materially useful fact: enforcement of Chapter 547 does not fall evenly across the market's price spectrum.

## 11. Do Intangible and Professionalization Signals Explain the Licensing Premium?

Section 10.2 established that licensed and unlicensed listings have structurally different pricing relationships, and that licensed listings average roughly double the price. That test can't say *why* — three candidate channels were raised at the time: a **compositional** effect (licensed listings skew toward pricier room types), a **trust/legitimacy** effect (a registration number as a signal to guests), and a **professionalization** effect (licensed operators investing more in furnishing, service, and presentation).

This section tests that directly: we add explicit trust and professionalization signals to the model and see whether the `licensed` coefficient **shrinks** once they're controlled for. If it shrinks a lot, the licensing premium is mostly proxying for something else we can now measure directly. If it barely moves, the premium is more structural than these specific signals can explain.

### 11.1 Engineering trust and professionalization signals

Three fields would be the obvious first choice for "trust": `host_response_rate`, `host_response_time`, and `host_acceptance_rate`.

In [48]:
responsiveness_cols = ['host_response_rate', 'host_response_time', 'host_acceptance_rate', 'host_verifications']
for col in responsiveness_cols:
    print(col, '-> missing:', listings[col].isna().sum(), f"({listings[col].isna().mean()*100:.1f}%)")

host_response_rate -> missing: 15003 (100.0%)
host_response_time -> missing: 15003 (100.0%)
host_acceptance_rate -> missing: 15003 (100.0%)
host_verifications -> missing: 15003 (100.0%)


**Insights:** the entire host-responsiveness cluster — response rate, response time, acceptance rate, and even `host_verifications` — is **100% missing**. Rather than force a variable that isn't there, we substitute two signals that *are* well-populated and map onto the same two channels:

- **Trust/legitimacy:** `host_identity_verified` (ID-verified badge) — well-populated, though extremely imbalanced (see below).
- **Professionalization:** `host_listings_count` — the number of listings a host manages. A host running dozens of units is a fundamentally different kind of operator than someone renting a spare room, independent of whether either one is licensed.
- We also add `amenities_count` (parsed from the `amenities` list) as a direct, structured proxy for how much a host has invested in the unit itself — the closest thing in this dataset to a "quality of offering" signal.

In [49]:
import ast

listings['amenities_count'] = listings['amenities'].apply(lambda s: len(ast.literal_eval(s)) if pd.notna(s) else 0)
listings['host_identity_verified_bin'] = (listings['host_identity_verified'] == 't').astype(int)
listings['log_host_listings_count'] = np.log(listings['host_listings_count'].clip(lower=1))

print('amenities_count:', listings['amenities_count'].describe().round(1).to_dict())
print('host_identity_verified rate:', listings['host_identity_verified_bin'].mean().round(3))
print('host_listings_count:', listings['host_listings_count'].describe().round(2).to_dict())

amenities_count: {'count': 15003.0, 'mean': 38.7, 'std': 14.7, 'min': 0.0, '25%': 29.0, '50%': 40.0, '75%': 48.0, 'max': 105.0}
host_identity_verified rate: 0.984
host_listings_count: {'count': 15003.0, 'mean': 7.12, 'std': 20.13, 'min': 1.0, '25%': 1.0, '50%': 2.0, '75%': 4.0, 'max': 399.0}


In [50]:
fig = px.box(listings, x='licensed_label', y='host_listings_count', color='licensed_label',
             template=PLOTLY_TEMPLATE, points=False, log_y=True,
             color_discrete_map={'Licensed': COLOR_MAIN, 'Unlicensed': COLOR_ACCENT},
             title='Host Portfolio Size (log scale): Licensed vs. Unlicensed Listings')
fig.update_layout(showlegend=False, yaxis_title='Host Listings Count (log scale)', xaxis_title='')
fig.show()

print(listings.groupby('licensed_label')['host_listings_count'].agg(['mean', 'median']).round(2))

                 mean  median
licensed_label               
Licensed        3.430   1.000
Unlicensed     14.340   3.000


**A real, somewhat counter-intuitive finding:** unlicensed listings belong to hosts with *larger* portfolios on average (mean 14.3 listings, median 3) than licensed listings (mean 3.4, median 1). This runs against the naive "licensed = professional operator" story. It actually makes sense given Chapter 547 itself: registration requires the unit to be the operator's **principal residence**, which structurally limits licensing to individual, small-scale hosts — a host managing dozens of units by definition can't claim all of them as a principal residence, so a large-portfolio operator is close to *definitionally* excluded from being licensed for most of their units. If anything, this suggests `licensed` leans toward proxying for "small-scale, owner-occupied" rather than "large-scale, professionally managed."

### 11.2 Does the licensed coefficient shrink once we control for these?

We refit the full-sample log-price model from Section 10 with these three variables added, and compare the `licensed` coefficient before and after.

In [51]:
intangible_cols = ['amenities_count', 'host_identity_verified_bin', 'log_host_listings_count']
x_cols_extended = x_cols + intangible_cols

extended_df = model_df.copy()
extended_df['amenities_count'] = listings['amenities_count'].values
extended_df['host_identity_verified_bin'] = listings['host_identity_verified_bin'].values
extended_df['log_host_listings_count'] = listings['log_host_listings_count'].values

X_extended = sm.add_constant(extended_df[x_cols_extended])
model_extended_log = sm.OLS(extended_df['log_price'], X_extended).fit()

licensed_before = model_full_log.params['licensed']
licensed_after = model_extended_log.params['licensed']
shrinkage_pct = (1 - licensed_after / licensed_before) * 100

print(f"licensed coefficient -- before controls: {licensed_before:.4f}  "
      f"(+{(np.exp(licensed_before)-1)*100:.1f}% price effect)")
print(f"licensed coefficient -- after controls:  {licensed_after:.4f}  "
      f"(+{(np.exp(licensed_after)-1)*100:.1f}% price effect)")
print(f'Coefficient shrinkage: {shrinkage_pct:.1f}%')
print(f"\nR2 before: {model_full_log.rsquared:.4f}   R2 after: {model_extended_log.rsquared:.4f}")

print('\nNew variables:')
new_var_tbl = model_extended_log.params[intangible_cols].to_frame('coef')
new_var_tbl['pct_effect'] = (np.exp(new_var_tbl['coef']) - 1) * 100
new_var_tbl['p_value'] = model_extended_log.pvalues[intangible_cols]
print(new_var_tbl.round(4))

licensed coefficient -- before controls: 0.3118  (+36.6% price effect)
licensed coefficient -- after controls:  0.2697  (+31.0% price effect)
Coefficient shrinkage: 13.5%

R2 before: 0.7288   R2 after: 0.7321

New variables:
                             coef  pct_effect  p_value
amenities_count             0.001       0.082    0.004
host_identity_verified_bin -0.018      -1.765    0.549
log_host_listings_count    -0.050      -4.825    0.000


**Insights:** the `licensed` coefficient shrinks by roughly **13%** (from a +37% price effect to a +31% price effect) once portfolio size, identity verification, and amenity count are controlled for — a real but *partial* explanation. `log_host_listings_count` is the variable doing the work here: a significant negative coefficient, meaning larger-portfolio hosts price *lower*, holding size and location fixed, consistent with the pattern above. `amenities_count` is statistically significant but tiny in practical size. `host_identity_verified` is not significant at all — unsurprising given 98.4% of hosts are ID-verified, leaving almost no variation for the model to use.

These three signals explain part of the licensing premium, but most of it survives even after controlling for them (still highly significant, still a sizeable effect). The Chow test's structural difference is not simply an artifact of portfolio size or amenity richness — something about licensing status itself, or something not captured by these three variables, still matters.

### 11.3 Are the new variables jointly significant?

Same block-testing logic as Section 10.1: rather than reading the three individual t-tests above in isolation, we test whether the block as a whole improves the model.

In [52]:
param_names_ext = X_extended.columns.tolist()
R_intangible = np.zeros((len(intangible_cols), len(param_names_ext)))
for i, col in enumerate(intangible_cols):
    R_intangible[i, param_names_ext.index(col)] = 1

wald_intangible = model_extended_log.wald_test(R_intangible)
print(wald_intangible)

<F test: F=array([[61.73301317]]), p=1.1658835275216727e-39, df_denom=1.5e+04, df_num=3>


**Insights:** the block is jointly significant (p ≈ 0) — even though `host_identity_verified` alone wasn't, the three variables together add real explanatory power, driven mainly by portfolio size.

**Revisiting the Section 10.2 conclusion:** the licensing premium is not purely compositional or purely a trust signal — it's a mix, and it's still mostly unexplained by what we can measure here. The most interesting finding in this section is arguably not the shrinkage itself, but the *direction* of the portfolio-size effect: it complicates a simple "licensed hosts are more professional" narrative and instead points toward Chapter 547's principal-residence rule mechanically sorting small-scale operators into the licensed pool and large-scale operators out of it. That's a regulatory-design finding as much as a pricing one, and it's worth stating plainly in any policy discussion of this result.

## 12. Out-of-Sample Validation

Everything so far has measured how well the model *explains* the data it was fit on. Here we check how well it *predicts* the 20% of listings held out in Section 5 and never used to fit any model above — the real test of whether this would be useful to a host pricing a *new* listing.

In [53]:
X_test = sm.add_constant(test_df[x_cols])
y_test_price = test_df['price']

# Linear model, trained on the training set (Section 6)
pred_linear = model1.predict(X_test)
rmse_linear = np.sqrt(np.mean((y_test_price - pred_linear) ** 2))
mae_linear = np.mean(np.abs(y_test_price - pred_linear))

# Log model, trained on the training set (Section 8) -- back-transform predictions to dollars
pred_log = np.exp(model1_log.predict(X_test))
rmse_log = np.sqrt(np.mean((y_test_price - pred_log) ** 2))
mae_log = np.mean(np.abs(y_test_price - pred_log))

results_compare = pd.DataFrame({
    'Model': ['Linear (Model 1)', 'Log-linear (Model 1b)'],
    'Test RMSE ($)': [rmse_linear, rmse_log],
    'Test MAE ($)': [mae_linear, mae_log],
}).round(2)
results_compare

,Model,Test RMSE ($),Test MAE ($)
0,Linear (Model 1),357.000,115.640
1,Log-linear (Model 1b),358.150,98.840


In [54]:
fig = go.Figure()
fig.add_trace(go.Bar(name='RMSE', x=results_compare['Model'], y=results_compare['Test RMSE ($)'],
                      marker_color=COLOR_MAIN))
fig.add_trace(go.Bar(name='MAE', x=results_compare['Model'], y=results_compare['Test MAE ($)'],
                      marker_color=COLOR_ACCENT))
fig.update_layout(template=PLOTLY_TEMPLATE, barmode='group', height=420,
                   title='Out-of-Sample Prediction Error: Linear vs. Log-Linear Model',
                   yaxis_title='Error ($ per night)')
fig.show()

**Insights:** RMSE is nearly identical between the two models — because RMSE squares errors, it's dominated by the handful of very expensive listings in the test set, and being wrong by a large dollar amount on a CA5,000/night listing swamps everything else regardless of which model made the prediction. **MAE tells a different, more decision-relevant story:** the log-linear model's *typical* prediction error is meaningfully lower. For the overwhelming majority of hosts — pricing a listing anywhere near the CA100–400/night range where most of the market actually sits — **the log-linear model is the more accurate and more useful one**.

## 13. Interactive Summary Dashboard

A single view a host or analyst could use to explore price by neighbourhood and room type together. Mean and median are shown side by side on a shared color scale.

In [55]:
# Restrict to the two dominant, well-populated room types -- Hotel room (43 listings) and
# Shared room (18 listings) produce sparse, noisy per-neighbourhood cells that stretch the
# color scale and make the rest of the heatmap harder to read.
main_room_types = ['Entire home/apt', 'Private room']
plot_data = listings[listings['room_type'].isin(main_room_types)]

pivot_mean = plot_data.pivot_table(index='neighbourhood_grp', columns='room_type',
                                    values='price', aggfunc='mean').loc[nbhd_order]
pivot_median = plot_data.pivot_table(index='neighbourhood_grp', columns='room_type',
                                      values='price', aggfunc='median').loc[nbhd_order]

# Shared color scale so the two panels are directly, visually comparable
shared_min = min(pivot_mean.min().min(), pivot_median.min().min())
shared_max = max(pivot_mean.max().max(), pivot_median.max().max())

fig = make_subplots(rows=1, cols=2, subplot_titles=('Mean Price ($)', 'Median Price ($)'),
                     shared_yaxes=True, horizontal_spacing=0.15)

fig.add_trace(go.Heatmap(
    z=pivot_mean.values, x=pivot_mean.columns, y=pivot_mean.index,
    colorscale='Blues', zmin=shared_min, zmax=shared_max,
    text=pivot_mean.values, texttemplate='$%{text:.0f}', textfont={'size': 11},
    colorbar=dict(title='Price ($)', x=1.02),
), row=1, col=1)

fig.add_trace(go.Heatmap(
    z=pivot_median.values, x=pivot_median.columns, y=pivot_median.index,
    colorscale='Blues', zmin=shared_min, zmax=shared_max,
    text=pivot_median.values, texttemplate='$%{text:.0f}', textfont={'size': 11},
    showscale=False,
), row=1, col=2)

fig.update_layout(template=PLOTLY_TEMPLATE, height=600,
                   title='Price by Neighbourhood x Room Type: Mean vs. Median, Side by Side')
fig.show()

## 14. Executive Summary and Recommendations

### For hosts: pricing guidance

- **Size and quality drive price predictably.** Each additional bathroom, bedroom, and guest capacity is associated with a statistically significant price premium (Section 8) — a host adding a bathroom or converting a den to a bedroom has a defensible basis for raising their nightly rate.
- **Adding sleeping capacity only pays off if it comes with more room.** More beds alone, without a matching increase in bedrooms or bathrooms, is associated with a *lower* price, not a higher one (Section 8) — packing more guests into the same footprint reads as a budget configuration, not a premium one. A host raising capacity should add a bedroom or bathroom alongside it, not just a mattress.
- **Room type is the single largest lever.** Entire-home listings command a large premium over private rooms; this is the biggest structural decision a host makes, well before fine-tuning.
- **Location matters, but unevenly.** The joint F-test (Section 10.1) confirms neighbourhood is a real, statistically significant driver of price — but not every neighbourhood differs meaningfully from the citywide baseline (the Annex). Hosts in typical, non-standout neighbourhoods shouldn't expect much of a locational premium or penalty; hosts in the highest- and lowest-priced pockets identified in Section 4.3 should price accordingly.
- **Minimum-night requirements come at a small per-night cost.** Hosts requiring longer minimum stays price a bit lower per night — a reasonable trade-off to flag explicitly rather than assume is free.
- **Overpricing has a measurable occupancy cost.** Listings priced well above what the model predicts for their characteristics see measurably lower occupancy. This is a modest effect next to size, room type, and location, but a real, testable version of the underpricing/overpricing tradeoff.
- **Using the log-linear model, not the raw-price model, for any pricing tool.** It fits dramatically better (R² 0.73 vs. 0.28) and predicts materially more accurately for typical listings (Section 12).

### For policy: the licensing finding

The Chow test in Section 10.2 found that licensed and unlicensed listings do not just differ in average price (\$331 vs. \$165) — **the entire pricing relationship differs structurally** between the two segments. Combined with the fact that over a third of listings in this dataset display no registration number, this suggests Toronto's short-term rental market is not one uniform market with a compliance overlay, but something closer to two distinct markets operating side by side. For Municipal Licensing and Standards, this is a useful data point for prioritizing enforcement: unlicensed activity is concentrated disproportionately at the lower end of the price spectrum, which may correspond to a different type of operator (occasional/informal hosts) than the higher-priced, licensed, likely more professionally managed segment.

## 15. Limitations and Scope

- **Single snapshot, not a time series.** All data reflects one Inside Airbnb scrape (May–June 2026). `calendar.csv` in this snapshot records availability but not per-night price, so no seasonal pricing analysis was possible — a genuine data constraint, not an oversight.
- **`licensed` is a proxy for current compliance status, not a before/after comparison.** The Chow test compares listings that currently display a registration number against those that don't; it cannot speak to whether any individual listing's pricing changed *when* it became licensed.
- **Imputed, not observed, size variables.** `bedrooms`, `beds`, and `bathrooms` were imputed for a substantial share of listings (26–54% missing before imputation). Results are conditional on the property-type-median imputation strategy used in Section 3.5.
- **Neighbourhood grouping is a modeling choice.** Collapsing 140 neighbourhoods to the top 14 + "Other" was necessary for a stable, interpretable dummy-variable block, but it does mean some finer-grained locational variation within the "Other" bucket is not captured.
- **No text mining.** The 576,000+ review comments were not analyzed; only structured review counts, scores, and dates were used.
- **Correlational, not causal.** Every finding in this notebook — including the licensing result — describes an association the data supports, not a causal claim about what would happen if a specific listing's characteristics or licensing status changed.
- **Toronto only.** The model reflects Toronto's specific market and regulatory environment (Chapter 547) and is not assumed to generalize to other cities.